## Data Ingestion & Initial Exploratory Data Analysis (EDA)

In this phase, we set up our core environment, ingest the raw Titanic dataset, and execute foundational diagnostic checks to analyze the structure, integrity, and missingness of the data.


In [107]:
import pandas as pd
import numpy as np

df=pd.read_csv('../data/raw/Titanic-Dataset.csv')
# Diagnostic Commands
print(df.info())
print(df.isnull().sum()/len(df)*100)
print(df.head())
df.describe()

<class 'pandas.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    str    
 4   Sex          891 non-null    str    
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    str    
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    str    
 11  Embarked     889 non-null    str    
dtypes: float64(2), int64(5), str(5)
memory usage: 118.9 KB
None
PassengerId     0.000000
Survived        0.000000
Pclass          0.000000
Name            0.000000
Sex             0.000000
Age            19.865320
SibSp           0.000000
Parch           0.000000
Ticket          0.000000
Fare            0.000000
Cab

,PassengerId,Survived,Pclass,Age,SibSp,Parch,Fare
count,891.000000,891.000000,891.000000,714.000000,891.000000,891.000000,891.000000
mean,446.000000,0.383838,2.308642,29.699118,0.523008,0.381594,32.204208
std,257.353842,0.486592,0.836071,14.526497,1.102743,0.806057,49.693429
min,1.000000,0.000000,1.000000,0.420000,0.000000,0.000000,0.000000
25%,223.500000,0.000000,2.000000,20.125000,0.000000,0.000000,7.910400
50%,446.000000,0.000000,3.000000,28.000000,0.000000,0.000000,14.454200
75%,668.500000,1.000000,3.000000,38.000000,1.000000,0.000000,31.000000
max,891.000000,1.000000,3.000000,80.000000,8.000000,6.000000,512.329200


# Data Preprocessing & Advanced Feature Engineering

In this stage, we execute targeted data cleaning and transformation steps to handle missing values, and preserve structural signals.

### Preprocessing Actions & Technical Justifications

*   **Imputing `Age` by `Pclass` and `Sex` Group Median**
    *   *Action:* Calculate the median age for each distinct passenger class and gender combination, then fill missing ages using their respective group's median.
    *   *Reasoning:* Passenger age strongly correlates with socioeconomic status (class) and gender roles of the era. Imputing by group median preserves these demographic distributions far better than applying a global dataset average.
*   **Creating the `Has_Cabin` Indicator Column**
    *   *Action:* Convert the sparse text in the `Cabin` column into a binary flag (`1` if a cabin number exists, `0` if missing).
    *   *Reasoning:* 77% of the cabin data is missing. However, the *presence* of an assigned cabin strongly correlates with 1st-class tickets and higher survival rates, transforming a data gap into a highly predictive visual feature.
*   **Dropping the Original `Cabin` Column**
    *   *Action:* Permanently remove the original high-cardinality `Cabin` text column using `axis=1`.
    *   *Reasoning:* The raw cabin codes contain too many missing values and unique strings (high cardinality), which would cause overfitting or structural noise in traditional classification models.
*   **Imputing `Embarked` with the Dataset Mode**
    *   *Action:* Fill the small number of missing ports in the `Embarked` column with the most frequently occurring value (the mode).
    *   *Reasoning:* With only a couple of values missing, using the dataset's most common boarding location (`'S'` for Southampton) resolves the nulls efficiently without introducing statistical bias.

In [108]:
# Fill missing ages with median of their respctive groups
df['Age_Filled']=df.groupby(['Pclass','Sex'])['Age'].transform(lambda x: x.fillna(x.median()))
# Extract value from Cabin before dropping it
df['Has_Cabin']=df['Cabin'].notnull().astype(int)
# Impute Embarked with its mode value
df['Embarked']=df['Embarked'].fillna(df['Embarked'].mode()[0])
# Drop cabin column
df=df.drop('Cabin',axis=1)
# Check filled data
df.head()


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Embarked,Age_Filled,Has_Cabin
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,S,22.0,0
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C,38.0,1
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,S,26.0,0
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,S,35.0,1
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,S,35.0,0


# Advanced Feature Engineering & Domain-Specific Interactions
In this step, we extract latent signals from passenger data by engineering compounding features. These domain-specific modifications help capture social structures and passenger dynamics.

### Feature Engineering Roadmap & Justifications

*   **`FamilySize` & `FamilyBand` (Social Stratification)**
    *   On the Titanic, survival rates varied heavily by family dynamics. Small families could coordinate quickly and stick together, large families faced bottleneck friction trying to find everyone, and solitary passengers had no dependent ties.
*   **`Fare_Per_Person` (True Economic Power)**
    *   The raw `Fare` value represents the cost of the *entire ticket group*, not the individual. Normalizing it reveals the actual socioeconomic power of each individual passenger.
*   **`AgeGroup` (Prioritization Proxy)**
    *   This captures the "women and children first" historical evacuation protocol directly as a structural indicator variable.
*   **`Title` Extraction (Societal Standing)**
    *   Titles offer a dense signal combining age, gender, social prestige, and marital status, which helps capture underlying survival variance that raw numbers miss.
*   **`SexEncoded` (Algorithmic Compatibility)**
    *   Map text values (`male`, `female`) to numeric identifiers (`0`, `1`). This transforms our most critical predictive category into an mathematically readable binary format.

In [117]:
# Compute total family count on board
df['FamilySize']=df['SibSp']+df['Parch']+1
# Caculate fare per person by dividing fare to family size 
df['Fare_Per_Person']=df['Fare']/df['FamilySize']
# Create a helper function to group family sizes into behavioral categories
def familyband(s):
    if s==1:
        return 'Alone'
    elif s<=4:
        return 'Small'
    else:
        return 'Large'
# Apply the grouping logic to convert numerical sizes into meaningful categorical bands
df['FamilyBand']=df['FamilySize'].apply(familyband)
# Use vectorized numpy.where to quickly tag passengers as 'Adult' (>= 18) or 'Child' (< 18)
df['AgeGroup']=np.where(df['Age_Filled']>=18,'Adult','Child')
# Extract string chunks from 'Name' by splitting on comma and period to isolate title
df['Title']=df['Name'].str.split(',').str[1].str.split('.').str[0].str.strip()
# Binarize gender into numerical values (0 for male, 1 for female
df['SexEncoded']=df['Sex'].map({'male':0,'female':1})
df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Embarked,Age_Filled,Has_Cabin,FamilySize,Fare_Per_Person,FamilyBand,AgeGroup,Title,SexEncoded
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,S,22.0,0,2,3.62500,Small,Adult,Mr,0
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C,38.0,1,2,35.64165,Small,Adult,Mrs,1
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,S,26.0,0,1,7.92500,Alone,Adult,Miss,1
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,S,35.0,1,2,26.55000,Small,Adult,Mrs,1
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,S,35.0,0,1,8.05000,Alone,Adult,Mr,0


# Slicing the Data for Specific Passenger Groups
Here, we are slicing the dataset to pull exact headcounts for specific, high interest demographics. This helps us verify our data and answer historical questions about the disaster.

### What We Are Measuring
*   **1st-Class Females:** 
    *   We are pulling the exact headcount of women traveling in first class. Because this demographic historically had the highest survival rate, we need to know the baseline number of passengers in this group.
*   **Children Who Did Not Survive:** 
    *   We are filtering for passengers under 18 whose `Survived` flag is `0`. This helps us test if the "women and children first" rule actually held true across all passenger classes.

In [ ]:
# Use .query() to filter for women in 1st class and grab the total row count with .shape[0]
first_class_female=df.query("Sex=='female' and Pclass==1").shape[0]
print(f"The count of females from 1st class are {first_class_female}")
# Use boolean indexing (&) to find passengers under 18 who did not survive
children_unsurvived=df[(df['Age_Filled']<18) & (df['Survived']==0)].shape[0]
print(f"The total children who could not survived are {children_unsurvived}")

The count of females from 1st class are 94
The total children who could not survived are 52


# Calculating Survival Rates Across Groups

In this section, we are grouping our data to find out exactly which factors like gender, class, or where a passenger boarded  influenced their chances of survival.

### What We Are Measuring
*   **Survival Rates by Class and Gender:**
    *   We group the data by `Pclass` and `Sex` to see the exact percentage of people who survived in each segment (e.g., 1st class women vs. 3rd-class men).
*   **Survival Rates by Class and Boarding Port (Pivot Table):**
    *   We create a pivot table to cross reference passenger class against their onboarding port (`Embarked`). Adding `margins=True` gives us the overall averages (the "All" row and column) so we can spot any broader trends easily.

In [111]:
# Group by Class and Sex, calculate the survival average, and multiply by 100 to get percentages
print(df.groupby(['Pclass','Sex'])['Survived'].mean()*100)
# Build a pivot table showing survival rates across Class vs. Onboarding Port
pivot=df.pivot_table(
    index='Pclass',
    columns='Embarked',
    values='Survived',
    aggfunc='mean',
    margins=True
)
pivot

Pclass  Sex   
1       female    96.808511
        male      36.885246
2       female    92.105263
        male      15.740741
3       female    50.000000
        male      13.544669
Name: Survived, dtype: float64


Embarked,C,Q,S,All
Pclass,,,,
1,0.694118,0.500000,0.589147,0.629630
2,0.529412,0.666667,0.463415,0.472826
3,0.378788,0.375000,0.189802,0.242363
All,0.553571,0.389610,0.339009,0.383838


# Verifying Group Headcounts (Sanity Check)

In this section, we create a matching count table for our previous survival analysis. This allows us to check the raw sample size of each group to make sure our survival percentages are trustworthy.

### What We Are Measuring
*   **Total Passenger Counts by Class and Port:**
    *   Instead of looking at percentages, we use `aggfunc='count'` to see the actual number of people inside each group. 
    *   *Why this matters:* If a specific group (like 1st-class passengers who boarded at port 'Q') has a 100% survival rate but a count of only 2 people, we know that rate might just be a random fluke rather than a reliable trend.

In [112]:
# Build a matching count pivot table to check the underlying sample sizes
df.pivot_table(
    index='Pclass',
    columns='Embarked',
    values='Survived',
    aggfunc='count',
    margins=True
)


Embarked,C,Q,S,All
Pclass,,,,
1,85,2,129,216
2,17,3,164,184
3,66,72,353,491
All,168,77,646,891


# Spotting High-Value Passengers

Here, we isolate the top 5 individual passengers who paid the absolute highest fare per person to see if their luxury ticket investment influenced their survival status.

### What We Are Measuring
*   **Top 5 Highest `Fare_Per_Person` Payers:**
    *   We use the `.nlargest()` function to quickly sort and pull the top 5 passengers who paid the most money per family member. We want to check their passenger class (`Pclass`) and whether they ultimately survived (`Survived`) to inspect the correlation between high wealth and emergency prioritization.

In [113]:
# Pull the 5 passengers with the highest calculated Fare_Per_Person values
highest_paid_customer=df.nlargest(5,'Fare_Per_Person')
highest_paid_customer

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Embarked,Age_Filled,Has_Cabin,FamilySize,Fare_Per_Person,FamilyBand,AgeGroup,Title,SexEncoded
258,259,1,1,"Ward, Miss. Anna",female,35.0,0,0,PC 17755,512.3292,C,35.0,0,1,512.3292,Alone,Adult,Miss,1
737,738,1,1,"Lesurer, Mr. Gustave J",male,35.0,0,0,PC 17755,512.3292,C,35.0,1,1,512.3292,Alone,Adult,Mr,0
679,680,1,1,"Cardeza, Mr. Thomas Drake Martinez",male,36.0,0,1,PC 17755,512.3292,C,36.0,1,2,256.1646,Small,Adult,Mr,0
380,381,1,1,"Bidois, Miss. Rosalie",female,42.0,0,0,PC 17757,227.5250,C,42.0,0,1,227.5250,Alone,Adult,Miss,1
557,558,0,1,"Robbins, Mr. Victor",male,NaN,0,0,PC 17757,227.5250,C,40.0,0,1,227.5250,Alone,Child,Mr,0


# Unpivoting Data for Easier Visualization

Here, we are reshaping our survival pivot table by flattening it out. This makes the data much easier to feed into visualization libraries like Seaborn or Plotly later on.

### What We Are Doing (Data Melting)
*   **Reshaping the Pivot Table:**
    *   We are using `pd.melt()` to turn wide data into long data. Instead of having the boarding ports (`C`, `Q`, `S`) spread out as separate columns, we collapse them into a single row by row format under an `Embarked` column with their matching `Survival_Rate` right next to them.

In [114]:
pd.melt(
    pivot.reset_index(),
    id_vars='Pclass',
    value_vars=['C','Q','S'],
    var_name='Embarked',
    value_name='Survival_Rate'
)

,Pclass,Embarked,Survival_Rate
0,1,C,0.694118
1,2,C,0.529412
2,3,C,0.378788
3,All,C,0.553571
4,1,Q,0.500000
5,2,Q,0.666667
6,3,Q,0.375000
7,All,Q,0.389610
8,1,S,0.589147
9,2,S,0.463415


# Saving the Cleaned Dataset

Now that all the missing values are filled and new features are engineered, we export the clean DataFrame into a new CSV file. This keeps our original raw data untouched and gives us a clean file ready for modeling.

### What We Are Doing
*   **Exporting to CSV:** 
    *   We use `.to_csv()` to save the file. 
    *   We set `index=False` so Pandas doesn't accidentally save the row numbers (0, 1, 2...) as an extra, unnamed column in our new file.

In [ ]:
# Save the cleaned dataframe to a new CSV file without row index indices
df.to_csv('../data/processed/titanic_cleaned.csv',index=False)